In [45]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [46]:
spark = SparkSession.builder \
    .appName("Week05 Spark Assignment") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.2.0


In [47]:
df = spark.read.csv(
    r"D:\Celebal\Week05_Spark_Assignment\DATA\synthetic_online_retail_data.csv",
    header=True,
    inferSchema=True
)

In [48]:
print("="*60)
print("ORIGINAL DATA")
print("="*60)

df.show(5)

print("\nSchema")
df.printSchema()

print("\nRows :", df.count())
print("Columns :", len(df.columns))


ORIGINAL DATA
+-----------+----------+----------+-----------+------------------+------------+--------+------+--------------+--------------+------------+------+---+
|customer_id|order_date|product_id|category_id|     category_name|product_name|quantity| price|payment_method|          city|review_score|gender|age|
+-----------+----------+----------+-----------+------------------+------------+--------+------+--------------+--------------+------------+------+---+
|      13542|2024-12-17|       784|         10|       Electronics|  Smartphone|       2|373.36|   Credit Card|New Oliviaberg|         1.0|     F| 56|
|      23188|2024-06-01|       682|         50| Sports & Outdoors| Soccer Ball|       5|299.34|   Credit Card|  Port Matthew|        NULL|     M| 59|
|      55098|2025-02-04|       684|         50| Sports & Outdoors|        Tent|       5|  23.0|   Credit Card|    West Sarah|         5.0|     F| 64|
|      65208|2024-10-28|       204|         40|Books & Stationery|  Story Book|       

In [49]:
print("="*60)
print("NULL VALUES")
print("="*60)

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

NULL VALUES
+-----------+----------+----------+-----------+-------------+------------+--------+-----+--------------+----+------------+------+---+
|customer_id|order_date|product_id|category_id|category_name|product_name|quantity|price|payment_method|city|review_score|gender|age|
+-----------+----------+----------+-----------+-------------+------------+--------+-----+--------------+----+------------+------+---+
|          0|         0|         0|          0|            0|           0|       0|    0|             0|   0|         201|   103|  0|
+-----------+----------+----------+-----------+-------------+------------+--------+-----+--------------+----+------------+------+---+



In [50]:
df_clean = df.na.fill({
    "review_score":0,
    "gender":"Unknown"
})

print("Missing values handled.")

Missing values handled.


In [51]:
rows_before = df_clean.count()

df_clean = df_clean.dropDuplicates()

rows_after = df_clean.count()

print("Rows Before :", rows_before)
print("Rows After :", rows_after)

Rows Before : 1000
Rows After : 1000


In [52]:
print("="*60)
print("PRICE GREATER THAN 200")
print("="*60)

df_clean.filter(col("price") > 200).show(10)

PRICE GREATER THAN 200
+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+----------------+------------+-------+---+
|customer_id|order_date|product_id|category_id|     category_name|product_name|quantity| price|  payment_method|            city|review_score| gender|age|
+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+----------------+------------+-------+---+
|      63313|2024-08-09|       182|         50| Sports & Outdoors|        Tent|       5|266.16|   Bank Transfer|     Jeffreyview|         1.0|      F| 59|
|      43226|2024-09-09|       221|         10|       Electronics|  Headphones|       1|306.53|Cash on Delivery|     North David|         5.0|      M| 32|
|      79047|2024-09-16|       517|         50| Sports & Outdoors|  Basketball|       3|310.39|   Bank Transfer|      Rhodesfurt|         4.0|      M| 62|
|      24744|2024-07-08|       818|         10|

In [53]:
print("="*60)
print("FEMALE CUSTOMERS")
print("="*60)

df_clean.filter(col("gender")=="F").show(10)

FEMALE CUSTOMERS
+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+----------------+------------+------+---+
|customer_id|order_date|product_id|category_id|     category_name|product_name|quantity| price|  payment_method|            city|review_score|gender|age|
+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+----------------+------------+------+---+
|      63313|2024-08-09|       182|         50| Sports & Outdoors|        Tent|       5|266.16|   Bank Transfer|     Jeffreyview|         1.0|     F| 59|
|      36759|2024-04-16|       952|         40|Books & Stationery|       Novel|       5|155.25|Cash on Delivery|     Michaelfurt|         0.0|     F| 43|
|      44704|2025-02-25|       911|         30|     Home & Living|      Carpet|       4| 37.52|Cash on Delivery|      Mccannfurt|         0.0|     F| 62|
|      37401|2024-08-16|       861|         50| Sports & Ou

In [54]:
df_clean.select(
    "customer_id",
    "product_name",
    "price",
    "quantity"
).show(10, truncate=False)

+-----------+------------+------+--------+
|customer_id|product_name|price |quantity|
+-----------+------------+------+--------+
|63313      |Tent        |266.16|5       |
|38523      |Blanket     |40.44 |3       |
|43226      |Headphones  |306.53|1       |
|36759      |Novel       |155.25|5       |
|32747      |Basketball  |24.0  |1       |
|79047      |Basketball  |310.39|3       |
|24744      |Smartphone  |200.44|5       |
|90219      |Eraser      |405.65|1       |
|44704      |Carpet      |37.52 |4       |
|52435      |Smartphone  |340.31|5       |
+-----------+------------+------+--------+
only showing top 10 rows


In [55]:
print("="*60)
print("SORT BY PRICE")
print("="*60)

df_clean.orderBy(col("price").desc()).show(10)

SORT BY PRICE
+-----------+----------+----------+-----------+------------------+-------------+--------+------+----------------+----------------+------------+-------+---+
|customer_id|order_date|product_id|category_id|     category_name| product_name|quantity| price|  payment_method|            city|review_score| gender|age|
+-----------+----------+----------+-----------+------------------+-------------+--------+------+----------------+----------------+------------+-------+---+
|      21063|2025-01-23|       327|         20|           Fashion|        Shirt|       2| 499.5|   Bank Transfer|     Dorseymouth|         0.0|      M| 66|
|      82894|2024-12-15|       585|         20|           Fashion|        Shirt|       1|499.23|   Bank Transfer|        Johnstad|         5.0|Unknown| 58|
|      23420|2024-08-19|       508|         20|           Fashion|        Shirt|       4|498.66|     Credit Card|Port Adrianhaven|         5.0|      F| 56|
|      39427|2024-06-13|       292|         20|   

In [56]:
print("="*60)
print("GROUP BY CATEGORY")
print("="*60)

df_clean.groupBy("category_name") \
    .count() \
    .show()

GROUP BY CATEGORY
+------------------+-----+
|     category_name|count|
+------------------+-----+
|           Fashion|  198|
| Sports & Outdoors|  211|
|Books & Stationery|  193|
|       Electronics|  207|
|     Home & Living|  191|
+------------------+-----+



In [57]:
print("="*60)
print("AGGREGATION")
print("="*60)

df_clean.select(
    count("*").alias("Total Records"),
    sum("price").alias("Total Price"),
    avg("price").alias("Average Price"),
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price")
).show()

AGGREGATION
+-------------+------------------+------------------+-------------+-------------+
|Total Records|       Total Price|     Average Price|Minimum Price|Maximum Price|
+-------------+------------------+------------------+-------------+-------------+
|         1000|251850.66000000012|251.85066000000012|        10.72|        499.5|
+-------------+------------------+------------------+-------------+-------------+



In [58]:
print("="*60)
print("CATEGORY WISE SALES")
print("="*60)

df_clean.groupBy("category_name") \
    .agg(
        sum("price").alias("Total Price"),
        avg("price").alias("Average Price")
    ).show()

CATEGORY WISE SALES
+------------------+------------------+------------------+
|     category_name|       Total Price|     Average Price|
+------------------+------------------+------------------+
|           Fashion|48428.570000000036|244.58873737373756|
| Sports & Outdoors|52966.080000000016|251.02407582938397|
|Books & Stationery| 50386.77000000003| 261.0713471502592|
|       Electronics|53622.669999999984| 259.0467149758453|
|     Home & Living| 46446.56999999997| 243.1757591623035|
+------------------+------------------+------------------+



In [59]:
print("="*60)
print("TOTAL AMOUNT")
print("="*60)

df_clean = df_clean.withColumn(
    "Total_Amount",
    col("price") * col("quantity")
)

df_clean.show(10)

TOTAL AMOUNT
+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+------------------+------------+-------+---+------------------+
|customer_id|order_date|product_id|category_id|     category_name|product_name|quantity| price|  payment_method|              city|review_score| gender|age|      Total_Amount|
+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+------------------+------------+-------+---+------------------+
|      63313|2024-08-09|       182|         50| Sports & Outdoors|        Tent|       5|266.16|   Bank Transfer|       Jeffreyview|         1.0|      F| 59|1330.8000000000002|
|      38523|2024-06-20|       383|         30|     Home & Living|     Blanket|       3| 40.44|     Credit Card|       Rodgersfurt|         4.0|      M| 60|            121.32|
|      43226|2024-09-09|       221|         10|       Electronics|  Headphones|       1|306.53|Cash on Deli

In [60]:
df_clean.createOrReplaceTempView("retail")

In [61]:
spark.sql("""
SELECT
category_name,
SUM(price) AS Total_Price
FROM retail
GROUP BY category_name
""").show()

+------------------+------------------+
|     category_name|       Total_Price|
+------------------+------------------+
|           Fashion|48428.570000000036|
| Sports & Outdoors|52966.080000000016|
|Books & Stationery| 50386.77000000003|
|       Electronics|53622.669999999984|
|     Home & Living| 46446.56999999997|
+------------------+------------------+



In [62]:
spark.sql("""
SELECT *
FROM retail
WHERE price > 300
""").show()

+-----------+----------+----------+-----------+------------------+-------------+--------+------+----------------+----------------+------------+-------+---+------------+
|customer_id|order_date|product_id|category_id|     category_name| product_name|quantity| price|  payment_method|            city|review_score| gender|age|Total_Amount|
+-----------+----------+----------+-----------+------------------+-------------+--------+------+----------------+----------------+------------+-------+---+------------+
|      43226|2024-09-09|       221|         10|       Electronics|   Headphones|       1|306.53|Cash on Delivery|     North David|         5.0|      M| 32|      306.53|
|      79047|2024-09-16|       517|         50| Sports & Outdoors|   Basketball|       3|310.39|   Bank Transfer|      Rhodesfurt|         4.0|      M| 62|      931.17|
|      90219|2024-07-15|       430|         40|Books & Stationery|       Eraser|       1|405.65|Cash on Delivery|      West April|         5.0|      M| 58|

In [63]:
def customer_type(age):
    if age < 25:
        return "Young"
    elif age < 50:
        return "Adult"
    else:
        return "Senior"

customer_udf = udf(customer_type, StringType())

df_clean = df_clean.withColumn(
    "Customer_Type",
    customer_udf(col("age"))
)

df_clean.show(10)

+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+------------------+------------+-------+---+------------------+-------------+
|customer_id|order_date|product_id|category_id|     category_name|product_name|quantity| price|  payment_method|              city|review_score| gender|age|      Total_Amount|Customer_Type|
+-----------+----------+----------+-----------+------------------+------------+--------+------+----------------+------------------+------------+-------+---+------------------+-------------+
|      63313|2024-08-09|       182|         50| Sports & Outdoors|        Tent|       5|266.16|   Bank Transfer|       Jeffreyview|         1.0|      F| 59|1330.8000000000002|       Senior|
|      38523|2024-06-20|       383|         30|     Home & Living|     Blanket|       3| 40.44|     Credit Card|       Rodgersfurt|         4.0|      M| 60|            121.32|       Senior|
|      43226|2024-09-09|       221|         10|   

In [69]:
# Convert Spark DataFrame to Pandas
pdf = df_clean.toPandas()

# Save as CSV
pdf.to_csv(
    r"D:\Celebal\Spark_Output.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


In [68]:
spark